# RFM Customer Segmentation Analysis


In [21]:
import pandas as pd
from sqlalchemy import create_engine

# ---
# Why we are doing this:
# We are connecting directly to the DuckDB data warehouse file.
# This allows us to access the clean, tested, and transformed data
# from our dbt pipeline instead of working with raw, messy CSV files.
# This is the standard, professional workflow.
# ---

# Create a connection to the DuckDB database file
# The database file is located in the `dbt_warehouse` directory at the root of the project.
engine = create_engine('duckdb:///../dbt_warehouse/olist.duckdb')

# Write a simple SQL query to select everything from our final RFM model
query = "SELECT * FROM mart_rfm"

# Execute the query and load the results into a pandas DataFrame
rfm_df = pd.read_sql(query, engine)

# Display the first few rows to verify the data was loaded correctly

rfm_df[rfm_df['frequency'] > 5].head(10)



,customer_unique_id,recency,frequency,monetary
320,63cfc61cee11cbe306bff5857d00bfe4,93,6,826.32
8950,3e43e6105506432c953e165fb2acf44c,183,9,1172.66
9619,ca77025e7201e3b30c44b472ff346268,89,7,1122.72
26933,f0e310a6839dce9de1638e0fe5ab282a,146,6,540.69
38228,47c1a3033b8b77b3ab6e109eb4d5fdf3,217,6,944.21
55880,dc813062e0fc23409cd255f7f53c7074,6,6,1094.63
61683,1b6c7548a2a1f9037c1fd3ddfed95f33,196,7,959.01
67789,8d50f5eadf50201ccdcedfb9e2ac8455,9,15,879.27
76541,6469f99c1f9dfae7733b25662e7f1782,62,7,758.83
85584,12f5d6e1cbf93dafd9dcc19095df0b3d,601,6,110.72


## Calculate RFM Scores

Now that we have the raw RFM values, we need to score each customer on a scale of 1-5 for each metric. This allows us to easily group and compare customers. We will use quintiles (dividing the data into 5 equal parts) to create these scores.


In [22]:
# ---
# Why we are doing this:
# Raw R, F, and M values are on different scales. Scoring them from 1-5
# standardizes them, making it easy to compare and combine them.
# `pd.qcut` is used for R and M, but F (frequency) is often heavily skewed
# (many customers with 1 purchase), so we use a custom function for it.
# ---

# Create labels for our scores (1 is worst, 5 is best)
r_labels = range(5, 0, -1) # For Recency, lower is better, so we reverse the labels
fm_labels = range(1, 6)    # For Frequency and Monetary, higher is better

# Calculate R and M scores using quintiles
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], q=5, labels=r_labels, duplicates='drop').astype(int)
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'], q=5, labels=fm_labels, duplicates='drop').astype(int)

# Define a function to score frequency based on common business rules
def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x == 4:
        return 4
    else: # 5 or more purchases
        return 5

# Apply the function to the frequency column
rfm_df['F_score'] = rfm_df['frequency'].apply(frequency_score)

# ---
# Why we are doing this:
# The combined RFM score gives us a single, comparable metric for each customer.
# We treat the scores as strings and concatenate them (e.g., a customer with 5 for R, 1 for F, and 3 for M becomes '513').
# ---

# Combine the individual scores into a single RFM_Score
def join_rfm(x):
    return str(x['R_score']) + str(x['F_score']) + str(x['M_score'])

rfm_df['RFM_Score'] = rfm_df.apply(join_rfm, axis=1)

# Display the first few rows with the new scores
rfm_df.head()


,customer_unique_id,recency,frequency,monetary,R_score,M_score,F_score,RFM_Score
0,addec96d2e059c80c30fe6871d30d177,191,1,22.77,3,1,1,311
1,66cc90195ca44cc7ac6a1cd0e1e1e7b2,324,1,30.40,2,1,1,211
2,8d46223c91cbeb93e0930ca8bd8ffca2,276,1,171.32,2,4,1,214
3,27cf4b153010911a0957150255a6c6db,137,1,465.40,4,5,1,415
4,be1e99a0c57d7c3c699cfc4db26c8edf,29,1,40.27,5,1,1,511


### Analyzing the Frequency Distribution

Before settling on our scoring logic, let's analyze the distribution of the `frequency` column. This will help us understand why `pd.qcut` failed and validate if our static business rules are reasonable. This is a key "Lab" step to inform our "Factory" logic.


In [23]:
# ---
# Why we are doing this:
# We need to understand the shape of our data to make good modeling choices.
# `value_counts` will show us exactly how many customers made 1, 2, 3, etc. purchases.
# `describe` will give us the statistical summary, including the percentiles.
# ---

# Show the count of customers for each frequency value
print("Frequency Value Counts:")
print(rfm_df['frequency'].value_counts(normalize=True).head(10))
print("\\n-------------------------\\n")

# Show the statistical summary of the frequency
print("Frequency Statistical Summary:")
print(rfm_df['frequency'].describe())



Frequency Value Counts:
frequency
1     0.969997
2     0.027561
3     0.001939
4     0.000300
5     0.000096
6     0.000054
7     0.000032
9     0.000011
15    0.000011
Name: proportion, dtype: float64
\n-------------------------\n
Frequency Statistical Summary:
count    93357.000000
mean         1.033420
std          0.209099
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.000000
Name: frequency, dtype: float64


## Defining Customer Segments

Now that we have the RFM scores, we can map them to descriptive, human-readable segments. This is the final step in translating our raw data into actionable business insights. We will define a set of rules to group customers based on their R and F scores, which are often the most powerful indicators of customer loyalty and engagement.


In [24]:
# ---
# Why we are doing this:
# A segment name like "Best Customers" is far more intuitive for business users
# than a score like "555". This mapping is the final and most crucial step
# in making our analysis actionable. We use a regex-based mapping on the
# R and F scores, as this is a standard and effective way to define segments.
# ---

# Create a dictionary to map RFM scores to segment names
# We use regex to match patterns in the R and F scores
segment_map = {
    r'[4-5][4-5]': 'Best Customers',  # R=4-5, F=4-5
    r'5[1-3]': 'New Customers',      # R=5, F=1-3
    r'4[1-3]': 'Potential Loyalists', # R=4, F=1-3
    r'[1-3][4-5]': 'At Risk',           # R=1-3, F=4-5 (frequent but not recent)
    r'[1-3]3': 'Needs Attention',     # R=1-3, F=3
    r'[1-3][1-2]': 'Hibernating',       # R=1-3, F=1-2
    r'[4-5][1-3]': 'Promising'        # R=4-5, F=1-3 (recent but not frequent)
}


# Create a new 'Segment' column by mapping the RFM scores
rfm_df['Segment'] = rfm_df['R_score'].astype(str) + rfm_df['F_score'].astype(str)
rfm_df['Segment'] = rfm_df['Segment'].replace(segment_map, regex=True)

# Display the first few rows with the new Segment column
rfm_df.head()



,customer_unique_id,recency,frequency,monetary,R_score,M_score,F_score,RFM_Score,Segment
0,addec96d2e059c80c30fe6871d30d177,191,1,22.77,3,1,1,311,Hibernating
1,66cc90195ca44cc7ac6a1cd0e1e1e7b2,324,1,30.40,2,1,1,211,Hibernating
2,8d46223c91cbeb93e0930ca8bd8ffca2,276,1,171.32,2,4,1,214,Hibernating
3,27cf4b153010911a0957150255a6c6db,137,1,465.40,4,5,1,415,Potential Loyalists
4,be1e99a0c57d7c3c699cfc4db26c8edf,29,1,40.27,5,1,1,511,New Customers


## Analyzing & Visualizing the Segments

This is the final step where we analyze the customer segments we've created. We need to understand the size and value of each segment to inform business strategy. A clear visualization is the most effective way to communicate these findings to stakeholders.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ---
# Why we are doing this:
# We need to aggregate our data to understand the characteristics of each segment.
# By grouping by segment, we can calculate key metrics like the number of customers,
# and the average recency, frequency, and monetary value for each group.
# This summary table is the core of our final analysis.
# ---

# Calculate the size and average values for each segment
segment_analysis = rfm_df.groupby('Segment').agg({
    'customer_unique_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).rename(columns={'customer_unique_id': 'customer_count'}).sort_values(by='customer_count', ascending=False)

# ---
# Why we are doing this:
# A visual is worth a thousand words. This bar chart will allow stakeholders
# to immediately see which customer segments are the largest and most valuable.
# We will create two plots: one for the size of each segment, and one for the
# average monetary value.
# ---

# Set up the plot style
sns.set(style='whitegrid')

# Create the figure and axes
fig, axes = plt.subplots(2, 1, figsize=(12, 12))
fig.suptitle('Customer Segment Analysis', fontsize=16)

# Plot 1: Customer Count by Segment
sns.barplot(x=segment_analysis.index, y='customer_count', data=segment_analysis, ax=axes[0], palette='viridis')
axes[0].set_title('Number of Customers by Segment')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Average Monetary Value by Segment
sns.barplot(x=segment_analysis.index, y='monetary', data=segment_analysis, ax=axes[1], palette='plasma')
axes[1].set_title('Average Monetary Value by Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Average Monetary Value')
axes[1].tick_params(axis='x', rotation=45)

# Show the plot
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

# Display the summary table
segment_analysis
